In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


# =========================
# 0. OUTPUT FOLDER
# =========================
os.makedirs("outputs", exist_ok=True)


# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("kidney_disease_balanced.csv")

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)


# =========================
# 2. SPLIT TARGET
# =========================
y = df["Target"]
X = df.drop("Target", axis=1)


# =========================
# 3. DETECT COLUMNS
# =========================
cat_cols = X.select_dtypes(include=['str']).columns

if len(cat_cols) == 0:
    cat_cols = X.select_dtypes(include=['str']).columns

num_cols = X.columns.difference(cat_cols)


# =========================
# 4. PREPROCESSING
# =========================

# Impute
X[num_cols] = SimpleImputer(strategy='mean').fit_transform(X[num_cols])
X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

# Encode categorical
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])

# Encode target
y = LabelEncoder().fit_transform(y)

# Scale (optional but good practice)
X = StandardScaler().fit_transform(X)


# =========================
# 5. TRAIN MODEL
# =========================
model = RandomForestClassifier()
model.fit(X, y)


# =========================
# 6. FEATURE IMPORTANCE
# =========================
importances = model.feature_importances_

feature_names = df.drop("Target", axis=1).columns

df_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Sort
df_imp = df_imp.sort_values(by="Importance", ascending=False)


# =========================
# 7. SAVE TABLE
# =========================
df_imp.to_csv("outputs/RQ4_table.csv", index=False)

print("\nRQ4 TABLE (Top Features):\n")
print(df_imp.head(10))


# =========================
# 8. CREATE FIGURE
# =========================
plt.figure()

# Plot top 10 features
top_n = 10
plt.barh(
    df_imp["Feature"].head(top_n)[::-1],
    df_imp["Importance"].head(top_n)[::-1]
)

plt.title("RQ4: Top 10 Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")

plt.tight_layout()

# Save figure
plt.savefig("outputs/RQ4_figure.pdf")

# Show figure
plt.show()


# =========================
# 9. CONFIRM SAVE
# =========================
print("\nFiles saved in:", os.getcwd() + "/outputs/")